In [2]:
import pandas as pd
import numpy as np
import re
import os

In [3]:
#Load raw dataset
df = pd.read_csv(
    "../../data/raw/spams.csv",
    encoding="latin-1",
    header=0  
)

print(f"Shape: {df.shape}")
print(df.head())

Shape: (5572, 5)
     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  


In [4]:
# CELL 3: Drop unnecessary columns & rename

#change to only 2 columns: label and text
# keep only the two expected columns, regardless of current naming
if {"v1", "v2"}.issubset(df.columns):
    df = df[["v1", "v2"]].copy()
elif {"label", "text"}.issubset(df.columns):
    df = df[["label", "text"]].copy()
else:
    raise KeyError(f"Expected columns ['v1','v2'] or ['label','text'], found: {df.columns.tolist()}")

# change column names for better understanding
df.rename(columns={"v1": "label", "v2": "text"}, inplace=True)

print("Columns after cleaning:", df.columns.tolist())
print(df.head(3))

Columns after cleaning: ['label', 'text']
  label                                               text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...


In [5]:

#Check missing values & duplicates
print("Missing Values")
print(df.isnull().sum())

print(f"\nDuplicates")
print(f"Total duplicate rows: {df.duplicated().sum()}")

# Remove duplicates if any
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"\nShape after removing duplicates: {df.shape}")

Missing Values
label    0
text     0
dtype: int64

Duplicates
Total duplicate rows: 403

Shape after removing duplicates: (5169, 2)


In [6]:
#Encode labels  (ham=0, spam=1)

df["label"] = df["label"].map({"ham": 0, "spam": 1})

print("Label distribution after encoding:")
print(df["label"].value_counts())
print(f"\nSpam ratio: {df['label'].mean():.2%}")

Label distribution after encoding:
label
0    4516
1     653
Name: count, dtype: int64

Spam ratio: 12.63%


In [12]:
#Clean text
def clean_text(text):
    text = str(text).lower()                        # lowercase
    text = re.sub(r"http\S+|www\S+", "", text)      # remove URLs
    text = re.sub(r"\d+", "", text)                 # remove numbers
    text = re.sub(r"[^\w\s]", "", text)             # remove special characters
    text = re.sub(r"\s+", " ", text).strip()        # remove extra whitespace
    return text

df["text_clean"] = df["text"].apply(clean_text)

# Compare before and after
print("Original:", df["text"][2])
print("Cleaned: ", df["text_clean"][2])

Original: Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
Cleaned:  free entry in a wkly comp to win fa cup final tkts st may text fa to to receive entry questionstd txt ratetcs apply overs


In [13]:
#Add auxiliary features

df["text_length"]    = df["text"].apply(len)
df["word_count"]     = df["text_clean"].apply(lambda x: len(x.split()))
df["num_uppercase"]  = df["text"].apply(lambda x: sum(1 for c in x if c.isupper()))
df["num_exclamation"]= df["text"].apply(lambda x: x.count("!"))

print(df[["label", "text_length", "word_count", "num_uppercase", "num_exclamation"]].head(5))

   label  text_length  word_count  num_uppercase  num_exclamation
0      0          111          20              3                0
1      0           29           6              2                0
2      1          155          25             10                0
3      0           49          11              2                0
4      0           61          13              2                0


In [14]:
# CELL 8: Export processed dataset
os.makedirs("../../data/processed", exist_ok=True)

output_path = "../../data/processed/spam_processed.csv"
df.to_csv(output_path, index=False, encoding="utf-8")

print(f"Saved to: {output_path}")
print(f"Final shape: {df.shape}")
print(df.head(3))

Saved to: ../../data/processed/spam_processed.csv
Final shape: (5169, 7)
   label                                               text  \
0      0  Go until jurong point, crazy.. Available only ...   
1      0                      Ok lar... Joking wif u oni...   
2      1  Free entry in 2 a wkly comp to win FA Cup fina...   

                                          text_clean  text_length  word_count  \
0  go until jurong point crazy available only in ...          111          20   
1                            ok lar joking wif u oni           29           6   
2  free entry in a wkly comp to win fa cup final ...          155          25   

   num_uppercase  num_exclamation  
0              3                0  
1              2                0  
2             10                0  
